In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.decomposition import TruncatedSVD
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from collections import defaultdict
from datetime import datetime
import subprocess
import sys

required = [
    "numpy",
    "pandas",
    "torch",
    "scikit-learn",
    "gensim"
]

def check_and_install(package):
    try:
        __import__(package)
        print(f"✅ {package} is already installed.")
    except ImportError:
        print(f"⏳ Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

for pkg in required:
    # gensim은 import 이름이 그대로, scikit-learn은 sklearn으로 사용
    mod_name = "sklearn" if pkg == "scikit-learn" else pkg
    check_and_install(mod_name)

✅ numpy is already installed.
✅ pandas is already installed.
✅ torch is already installed.
✅ sklearn is already installed.
✅ gensim is already installed.


## 0. txt데이터 전처리

rating.txt,trustnetwork.txt를 열에 따라 재배치

In [4]:
import pandas as pd

def load_ratings(filepath):
    columns = [
        "UserID", "Product", "Category", "Rating",
        "Helpfulness", "Time", "Review"
    ]

    # 각 줄을 직접 파싱해서 7개까지만 자르기
    rows = []
    with open(filepath, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("::::")
            if len(parts) < 7:
                continue
            row = parts[:7]
            rows.append(row)

    df = pd.DataFrame(rows, columns=columns)

    # 형변환 및 전처리
    df["Time"] = pd.to_datetime(df["Time"], format="%d.%m.%Y", errors="coerce")
    df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
    df.dropna(subset=["UserID", "Product", "Rating", "Review"], inplace=True)
    df["UserID"] = df["UserID"].astype(str)

    return df

def load_trust(filepath):
    df = pd.read_csv(filepath, sep="::::", engine="python", names=["u", "v"], usecols=[0, 1])

    # 마지막에 빈 컬럼이 들어오는 경우 제거
    df = df[["u", "v"]].dropna()
    
    # 문자열 처리
    df["u"] = df["u"].astype(str)
    df["v"] = df["v"].astype(str)
    
    # 중복 제거
    trust_pairs = set([tuple(x) for x in df.values])
    return trust_pairs

# Load data
ratings = load_ratings("rating.txt")
trust_pairs = load_trust("trustnetwork.txt")

print(ratings.head(3))
print(list(trust_pairs)[:5])


    UserID               Product        Category  Rating   Helpfulness  \
0  5247778  Pyrex Oblong Roaster  House & Garden    40.0  very helpful   
1  5247778      Nozeroy (France)          Travel    40.0  very helpful   
2  5247778  Tesco Red UK Cabbage    Food & Drink    40.0  very helpful   

        Time                                             Review  
0 2011-09-17      I was looking on the Asda web site for som...  
1 2011-09-16      This summer while touring France we stayed...  
2 2011-09-17      I sometimes buy a red cabbage from Tesco. ...  
[('6265842', '6894799'), ('5238639', '5275136'), ('5019931', '19606'), ('5000858', '5019443'), ('5080978', '5085848')]


Games카테고리만 추출

In [ ]:
# 'Category'가 'Games'인 행만 필터링
one_category_ratings = ratings[ratings["Category"] == "Games"]

# 결과를 탭으로 구분된 텍스트 파일로 저장
one_category_ratings.to_csv("Games.txt", sep="\t", index=False)

# games_ratings.txt 파일을 읽어오기
games_df = pd.read_csv("Games.txt", sep="\t")
games_df


## 1. 📊 정적 선호도 - Ratings 기반 (Matrix Factorization)

In [ ]:
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

def static_rating_features(ratings_df, n_components=32, normalize_flag=True):
    ratings_df["Time"] = pd.to_datetime(ratings_df["Time"])

    # 사용자-아이템 행렬을 sparse matrix로 변환
    user_item_matrix = ratings_df.pivot_table(index="UserID", columns="Product", values="Rating", fill_value=0)
    sparse_matrix = csr_matrix(user_item_matrix.values)

    # SVD 수행 (scipy.sparse.linalg.svds 사용)
    u, s, vt = svds(sparse_matrix, k=n_components)

    # 정규화 (L2 정규화)
    if normalize_flag:
        # 각 벡터마다 정규화 (L2 정규화)
        norm = np.linalg.norm(u, axis=1, keepdims=True)
        u = u / (norm + 1e-8)

    # 결과는 u, s, vt로 나뉘어 나오며, u는 사용자 잠재 벡터를 포함함
    return dict(zip(user_item_matrix.index, u))

# 예시 데이터프레임 생성 (테스트용)
data = {
    "UserID": ["U1", "U1", "U2", "U2", "U3", "U3"],
    "Product": ["P1", "P2", "P1", "P3", "P2", "P3"],
    "Rating": [5, 3, 4, 2, 1, 5]
}
ratings_df = pd.DataFrame(data)

# 함수 호출해서 테스트
user_embeddings = static_rating_features(games_df, n_components=32)

# 결과 출력
print("유저 수:", len(user_embeddings))
print("U1 벡터:", user_embeddings[3338])

## 2. ✍️ 정적 선호도 - Reviews 기반 (Doc2Vec)

In [ ]:
import pandas as pd
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.preprocessing import normalize

def static_review_features(ratings_df, vector_size=32, normalize_flag=True):
    grouped = ratings_df.groupby("UserID")["Review"].apply(lambda x: " ".join(x)).reset_index()
    documents = [TaggedDocument(doc.split(), [str(uid)]) for uid, doc in zip(grouped["UserID"], grouped["Review"])]
    model = Doc2Vec(documents, vector_size=vector_size, window=5, min_count=2, workers=4, epochs=40)
    
    user_vectors = {uid: model.dv[str(uid)] for uid in grouped["UserID"]}

    if normalize_flag:
        user_vectors = {uid: normalize([vec], norm='l2')[0] for uid, vec in user_vectors.items()}  # L2 정규화

    return user_vectors

# 테스트용 데이터프레임
data = {
    "UserID": ["U1", "U1", "U2", "U3", "U3"],
    "Review": [
        "This game is great",
        "I love playing it",
        "Not bad but not great",
        "Terrible experience",
        "Won't play again"
    ]
}
ratings_df = pd.DataFrame(data)

# 함수 실행
user_review_embeddings = static_review_features(games_df, vector_size=16)

# 결과 출력
print("유저 수:", len(user_review_embeddings))
print("U1 벡터:", user_review_embeddings[3338])

## 3. ⏳ 동적 선호도 - 시간 순 리뷰 기반 LSTM

In [ ]:
from sklearn.preprocessing import normalize as sklearn_normalize
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import torch.nn as nn

# TimeSeriesLSTM 모델 정의
class TimeSeriesLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        output, (hn, _) = self.lstm(x)
        return hn.squeeze(0)  # [batch, hidden_dim]

# 동적 사용자 특성 추출 함수
def dynamic_user_features(ratings_df, doc2vec_model, hidden_dim=32, normalize_flag=True):
    user_groups = ratings_df.groupby("UserID")
    user_vectors = {}

    for uid, group in user_groups:
        group = group.sort_values("Time")
        vecs = []

        for review in group["Review"]:
            words = review.split()
            vec = doc2vec_model.infer_vector(words)
            vecs.append(vec)

        seq_tensor = torch.tensor(vecs).unsqueeze(0).float()  # [1, seq_len, input_dim]
        lstm_model = TimeSeriesLSTM(input_dim=len(vecs[0]), hidden_dim=hidden_dim)
        dynamic_vec = lstm_model(seq_tensor).detach().numpy()

        # L2 정규화
        if normalize_flag:
            dynamic_vec = sklearn_normalize(dynamic_vec, norm='l2')  # L2 정규화

        user_vectors[uid] = dynamic_vec

    return user_vectors

# 테스트용 리뷰 데이터 생성
data = {
    "UserID": [1, 1, 1, 2, 2, 3, 3, 3],
    "Product": ["A", "B", "C", "A", "B", "A", "B", "C"],
    "Rating": [5, 4, 3, 5, 4, 3, 5, 4],
    "Review": [
        "This product is amazing",
        "Good product, worth the price",
        "Not bad, could be better",
        "Excellent product, highly recommend",
        "Pretty good, could be improved",
        "Average, could be improved",
        "Great quality, would buy again",
        "Decent, but there are better options"
    ],
    "Time": [1, 2, 3, 1, 2, 1, 2, 3]
}

ratings_df = pd.DataFrame(data)

# Doc2Vec 모델 학습 (임시 모델)
documents = [TaggedDocument(doc.split(), [str(uid)]) for uid, doc in zip(ratings_df["UserID"], ratings_df["Review"])]
doc2vec_model = Doc2Vec(documents, vector_size=32, window=5, min_count=1, workers=4, epochs=40)

print('\n normalize_flag=False \n'+'*'*100)
# 동적 사용자 특성 추출
user_embeddings = dynamic_user_features(ratings_df, doc2vec_model, hidden_dim=32, normalize_flag=False)

for user_id, embedding in user_embeddings.items():
    print(f"User {user_id} embedding: {embedding}")

print('\n normalize_flag=True \n'+'*'*100)
# 동적 사용자 특성 추출
user_embeddings = dynamic_user_features(ratings_df, doc2vec_model, hidden_dim=32, normalize_flag=True)

for user_id, embedding in user_embeddings.items():
    print(f"User {user_id} embedding: {embedding}")

## 4. 🧠 Context-Aware Deep Multilayer Projection

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

class ContextMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, context_dim):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, context_dim)
        )

    def forward(self, x):
        return self.projection(x)

def generate_negative_samples(user_vectors, trust_pos, num_negatives=None):
    # user_vectors의 키를 사용하여 유효한 사용자 목록 생성
    users = list(user_vectors.keys())
    
    # trust_pos에서 신뢰 관계 쌍을 집합으로 생성
    trust_set = set((u, v) for u, v in trust_pos)
    
    # 가능한 모든 유효한 (u, v) 쌍을 생성하되, (u != v) 및 (u, v) 쌍이 trust_set에 없도록 함
    all_possible = [(u, v) for u in users for v in users if u != v and (u, v) not in trust_set]

    # num_negatives가 주어지지 않으면, 음성 샘플 수는 신뢰 관계 수와 같음
    if num_negatives is None:
        num_negatives = len(trust_pos)

    # 가능한 음성 샘플을 랜덤으로 섞고, 주어진 수만큼 선택
    random.shuffle(all_possible)
    trust_neg = all_possible[:num_negatives]
    
    return trust_neg

def evaluate_model(model, user_vectors, trust_pos, trust_neg, gamma=1.0):
    model.eval()
    y_true = []
    y_pred = []

    for u, v, label in trust_pos + trust_neg:
        if u not in user_vectors or v not in user_vectors:
            continue
        vec_u = torch.tensor(user_vectors[u], dtype=torch.float32)
        vec_v = torch.tensor(user_vectors[v], dtype=torch.float32)

        with torch.no_grad():
            pu = model(vec_u)
            pv = model(vec_v)
            sim = F.cosine_similarity(pu, pv, dim=0)
            prob = torch.sigmoid(gamma * sim).item()

        y_true.append(label)
        y_pred.append(prob)

    # 이진 클래스 기준으로 평가
    y_pred_label = [1 if p >= 0.5 else 0 for p in y_pred]

    precision = precision_score(y_true, y_pred_label)
    recall = recall_score(y_true, y_pred_label)
    f1 = f1_score(y_true, y_pred_label)
    auc = roc_auc_score(y_true, y_pred)

    print(f"[Evaluation]")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(f"AUC-ROC:   {auc:.4f}")

def train_single_context(user_vectors, trust_pos, trust_neg, epochs=10, gamma=1.0, lr=1e-3, neg_ratio=2.0):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    vec_dim = len(next(iter(user_vectors.values())))
    model = ContextMLP(vec_dim, 64, 32).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    
    # trust_pos와 trust_neg에서 user_vectors에 존재하는 사용자만 필터링
    trust_pos = [(str(u), str(v)) for u, v in trust_pos if str(u) in user_vectors and str(v) in user_vectors]
    trust_neg = [(str(u), str(v)) for u, v in trust_neg if str(u) in user_vectors and str(v) in user_vectors]
    
    
    # 레이블 추가
    trust_pos_labeled = [(u, v, 1) for u, v in trust_pos]
    trust_neg_labeled = [(u, v, 0) for u, v in trust_neg]

    num_pos = len(trust_pos_labeled)
    num_neg = int(num_pos * neg_ratio)

    if len(trust_neg_labeled) > num_neg:
        random.shuffle(trust_neg_labeled)
        trust_neg_labeled = trust_neg_labeled[:num_neg]
        print(f"[Neg Sampling] Using {num_neg} negative samples based on ratio {neg_ratio}")

    print('trust_pos_labeled : ',len(trust_pos_labeled),'  trust_neg_labeled : ',len(trust_neg_labeled))

    for epoch in range(epochs):
        total_loss = 0
        model.train()
        for i, (u, v, label) in enumerate(trust_pos_labeled + trust_neg_labeled):
            vec_u = torch.tensor(user_vectors[u], dtype=torch.float32).to(device)
            vec_v = torch.tensor(user_vectors[v], dtype=torch.float32).to(device)

            # 모델의 출력값 계산
            pu = model(vec_u)
            pv = model(vec_v)

            # 코사인 유사도 계산
            sim = F.cosine_similarity(pu, pv, dim=0)
            prob = torch.sigmoid(gamma * sim)

            # 타겟 값
            target = torch.tensor(label, dtype=torch.float32).to(device)
            
            # 손실 계산
            loss = (prob - target) ** 2
            total_loss += loss.item()

            # 역전파 및 최적화
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        print(f"[Epoch {epoch+1}] Total Loss: {total_loss:.4f}")

    # 학습 종료 후 평가
    evaluate_model(model, user_vectors, trust_pos_labeled, trust_neg_labeled, gamma)


## 🧩 전체 통합: 사용자 임베딩 생성 및 예시 실행

In [ ]:
# Load data
ratings = pd.read_csv("Games.txt", sep="\t")
print("1. 데이터 로드 완료")

trust_pairs = load_trust("trustnetwork.txt")
print("2. 신뢰 관계 데이터 로드 완료")

# 1-2. Static Features
rating_vecs = static_rating_features(ratings)
print("3. Static rating features 생성 완료")

review_vecs = static_review_features(ratings)
print("4. Static review features 생성 완료")

# 3. Train shared Doc2Vec model first
doc_model = Doc2Vec([TaggedDocument(r.split(), [i]) for i, r in enumerate(ratings["Review"])], vector_size=32, epochs=30)
print("5. Doc2Vec 모델 학습 완료")

# Dynamic Features
dyn_vecs = dynamic_user_features(ratings, doc_model)
print("6. Dynamic user features 생성 완료")

# concatnate
user_features = {}
for uid in ratings["UserID"].unique():
    r = rating_vecs.get(uid, np.zeros(32))
    rv = review_vecs.get(uid, np.zeros(32))
    d = dyn_vecs.get(uid, np.zeros(32)).flatten()  # flatten: LSTM 결과가 (1, D)이기 때문
    user_features[uid] = np.concatenate([r, rv, d])  # 96-dim
print("7. 최종 사용자 피처 결합 완료")

# save to feather
df_user_features = pd.DataFrame([
    {"UserID": uid, "concatenated_vector": user_features[uid]}
    for uid in user_features
])

df_user_features["concatenated_vector"] = df_user_features["concatenated_vector"].apply(lambda x: x.tolist())
df_user_features.to_feather("Games_concatenated.feather")


In [ ]:
import pandas as pd

# Feather 파일 로드
df = pd.read_feather("./Games_concatenated.feather")

# 리스트를 문자열로 변환 (예: [1.0, 2.0, 3.0] → "1.0,2.0,3.0")
df['vector_str'] = df['concatenated_vector'].apply(lambda x: ','.join(map(str, x)))

# 필요한 열만 선택하고 저장
df[['UserID', 'vector_str']].to_csv("Games_vectors.txt", sep='\t', index=False, header=False)


#확인
df_user_features = pd.read_feather("./Games_concatenated.feather")
print(df_user_features)


In [11]:
df = pd.read_feather("./Games_concatenated.feather")
user_vectors = {
    str(row['UserID']): np.array(row['concatenated_vector'])
    for _, row in df.iterrows()
}

trust_pos = load_trust("./trustnetwork.txt")
trust_neg = generate_negative_samples(user_vectors, trust_pos,num_negatives=len(trust_pos))


print('✅학습시작')
# 학습
'''
def train_single_context(user_vectors, trust_pos, trust_neg, epochs=10, gamma=1.0, lr=1e-3)
'''
train_single_context(user_vectors, trust_pos, trust_neg)


✅학습시작
[Neg Sampling] Using 50606 negative samples based on ratio 2.0
trust_pos_labeled :  25303   trust_neg_labeled :  50606
[Epoch 1] Total Loss: 15266.1782
[Epoch 2] Total Loss: 15525.4166
[Epoch 3] Total Loss: 15444.1514
[Epoch 4] Total Loss: 15389.3634
[Epoch 5] Total Loss: 15532.8435
[Epoch 6] Total Loss: 15501.2170
[Epoch 7] Total Loss: 15479.4853
[Epoch 8] Total Loss: 15585.1299
[Epoch 9] Total Loss: 15467.2835
[Epoch 10] Total Loss: 15417.9949
[Evaluation]
Precision: 0.3943
Recall:    0.5242
F1-score:  0.4501
AUC-ROC:   0.5798
